# 2.1 Stock Coverage Analysis

This notebook checks whether each stock has enough price data **while it was in the S&P 500**. Raw downloads come from `data/historical`. After membership is loaded, the shortened files are saved to `data/Filter_historical`; all later coverage checks use those filtered files.

## 1. Setup

In [18]:
from getpass import getpass
from pathlib import Path

import pandas as pd
import pandas_market_calendars as mcal

RAW_FOLDER = Path('../data/historical')
FILTERED_FOLDER = Path('../data/Filter_historical')
STUDY_START = '2015-01-01'
STUDY_END = '2026-08-24'
LOW_COVERAGE_LIMIT = 90

nyse = mcal.get_calendar('NYSE')
study_days = nyse.schedule(
    start_date=STUDY_START,
    end_date=STUDY_END
).index.normalize()

## 2. Load stock data

In [12]:
def load_stock(ticker, folder):
    stock = pd.read_csv(folder / f'{ticker.upper()}.csv')
    stock['Date'] = pd.to_datetime(stock['Date'], errors='coerce')
    return stock.sort_values('Date').reset_index(drop=True)

## 3. Check one raw download

Apple is a quick check that a raw file loads correctly. This is not the final coverage calculation because membership has not been applied yet.

In [13]:
aapl_raw = load_stock('AAPL', RAW_FOLDER)
print(f'Rows: {len(aapl_raw):,}')
print(f'First date: {aapl_raw["Date"].min().date()}')
print(f'Last date: {aapl_raw["Date"].max().date()}')
aapl_raw.head()

Rows: 2,927
First date: 2015-01-02
Last date: 2026-08-24


,Date,Adj Close,Close,High,Low,Open,Volume
0,2015-01-02,24.171759,27.332500,27.860001,26.837500,27.847500,212818400
1,2015-01-05,23.490803,26.562500,27.162500,26.352501,27.072500,257142000
2,2015-01-06,23.493010,26.565001,26.857500,26.157499,26.635000,263188400
3,2015-01-07,23.822439,26.937500,27.049999,26.674999,26.799999,160423600
4,2015-01-08,24.737741,27.972500,28.037500,27.174999,27.307501,237458000


## 4. Coverage workflow

A raw file may contain prices from before a stock entered the S&P 500 or after it left. Those extra dates can make actual coverage appear higher than theoretical coverage. We therefore filter first and look for low coverage afterward.

## 5. Load historical S&P 500 membership and shorten the data

The membership file tells us which stocks belonged to the index on each date. Dots in tickers are changed to hyphens to match Yahoo filenames.

In [14]:
MEMBERSHIP_URL = (
    'https://raw.githubusercontent.com/chinobing/'
    'historical_sp500_constituents/main/'
    'sp_500_historical_components.csv'
)

def ticker_set(ticker_text):
    return {
        ticker.strip().replace('.', '-').split(' ')[0]
        for ticker in ticker_text.split(',')
    }

membership = pd.read_csv(MEMBERSHIP_URL)
membership['date'] = pd.to_datetime(membership['date'])
membership = membership.sort_values('date').reset_index(drop=True)
membership['ticker_set'] = membership['tickers'].apply(ticker_set)
membership_dates = pd.DatetimeIndex(membership['date'])

print(
    f'Membership history: {membership["date"].min().date()} '
    f'to {membership["date"].max().date()}'
)
membership.tail()

Membership history: 1996-01-02 to 2026-09-01


,date,tickers,ticker_set
3865,2026-08-28,"A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP...","{IP, BKNG, DVN, KEY, ALB, IBM, RMD, FERG, A, U..."
3866,2026-08-29,"A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP...","{IP, BKNG, DVN, KEY, ALB, IBM, RMD, FERG, A, U..."
3867,2026-08-30,"A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP...","{IP, BKNG, DVN, KEY, ALB, IBM, RMD, FERG, A, U..."
3868,2026-08-31,"A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP...","{IP, BKNG, DVN, KEY, ALB, IBM, RMD, FERG, A, U..."
3869,2026-09-01,"A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP...","{IP, BKNG, DVN, KEY, ALB, IBM, RMD, FERG, A, U..."


### Create the membership-period files

This happens immediately after membership is loaded. Each output keeps only dates when its ticker belonged to the S&P 500. From this point onward, coverage analysis reads only from `Filter_historical`.

In [15]:
def get_membership_days(ticker):
    member_days = []

    for day in study_days:
        row_number = membership_dates.searchsorted(day, side='right') - 1
        if row_number >= 0 and ticker in membership.loc[row_number, 'ticker_set']:
            member_days.append(day)

    return pd.DatetimeIndex(member_days)

FILTERED_FOLDER.mkdir(parents=True, exist_ok=True)
membership_days_by_ticker = {}
files_saved = 0

for raw_path in sorted(RAW_FOLDER.glob('*.csv')):
    ticker = raw_path.stem.upper()
    stock = load_stock(ticker, RAW_FOLDER)
    member_days = get_membership_days(ticker)
    membership_days_by_ticker[ticker] = member_days

    stock_days = stock['Date'].dt.normalize()
    filtered_stock = stock[stock_days.isin(member_days)].copy()
    filtered_stock.to_csv(FILTERED_FOLDER / raw_path.name, index=False)
    files_saved += 1

print(f'Filtered files saved: {files_saved}')
print(f'Saved to: {FILTERED_FOLDER}')

Filtered files saved: 616
Saved to: ../data/Filter_historical


## 6. Find low coverage

Coverage is calculated only from filtered files and against expected membership days. The result includes stocks below 90% only. Stocks at or above 90% are not given another classification.

In [16]:
low_coverage_stocks = []

for filtered_path in sorted(FILTERED_FOLDER.glob('*.csv')):
    ticker = filtered_path.stem.upper()
    member_days = membership_days_by_ticker.get(
        ticker, pd.DatetimeIndex([])
    )

    if len(member_days) == 0:
        continue

    stock = load_stock(ticker, FILTERED_FOLDER)
    stock_days = pd.DatetimeIndex(
        stock['Date'].dropna().dt.normalize().unique()
    )
    downloaded_days = member_days.intersection(stock_days)
    coverage = len(downloaded_days) / len(member_days) * 100

    if coverage < LOW_COVERAGE_LIMIT:
        low_coverage_stocks.append({
            'Ticker': ticker,
            'Entry Date': member_days.min().date(),
            'Exit Date': member_days.max().date(),
            'Expected Membership Days': len(member_days),
            'Downloaded Membership Days': len(downloaded_days),
            'Coverage During Membership (%)': round(coverage, 2),
        })

RESULT_COLUMNS = [
    'Ticker', 'Entry Date', 'Exit Date',
    'Expected Membership Days', 'Downloaded Membership Days',
    'Coverage During Membership (%)',
]
low_coverage_df = pd.DataFrame(
    low_coverage_stocks, columns=RESULT_COLUMNS
).sort_values('Coverage During Membership (%)').reset_index(drop=True)
low_coverage_df.to_csv('../data/low_coverage_stocks.csv', index=False)

print(f'Stocks below {LOW_COVERAGE_LIMIT}% coverage: {len(low_coverage_df)}')
low_coverage_df

Stocks below 90% coverage: 17


,Ticker,Entry Date,Exit Date,Expected Membership Days,Downloaded Membership Days,Coverage During Membership (%)
0,APC,2015-01-02,2019-08-08,1158,0,0.00
1,SPLS,2015-01-02,2017-09-12,679,0,0.00
2,SBNY,2021-12-20,2023-03-14,309,0,0.00
3,NFX,2015-01-02,2019-02-14,1037,0,0.00
4,LB,2015-01-02,2021-08-02,1657,0,0.00
5,STI,2015-01-02,2019-12-06,1242,0,0.00
6,FB,2015-01-02,2022-06-08,1872,0,0.00
7,INFO,2017-06-02,2022-03-01,1195,0,0.00
8,CSRA,2015-11-30,2018-04-03,589,0,0.00
9,CAM,2015-01-02,2016-04-01,314,0,0.00


## 7. Review the result

Every row above is a low-coverage case after membership filtering. These stocks may need a better historical source or manual review before pair selection. The filtered files are also ready for the split-analysis notebook.

`APC` downloaded from Yahoo Finance today is no longer Anadarko Petroleum. Yahoo currently maps `APC` to ARKO Petroleum Corp. a completely different company. ARKO Petroleum only began trading on Nasdaq under `APC` on **February 12, 2026**,

FB becomes META at 2022-06-08. META/historical has the full record of FB+META starting from 2015. So FB can be deleted with META/historical replacing META/Filter_historical.

|  ticker | Historical S&P company     | What's happening now                                                                                                                              | Simple action                   |     |
| ----------- | -------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------- | --- |
| **SPLS**    | Staples                    | `SPLS` has been **reused** by a PIMCO ETF launched in Jan 2026.                                                                                   | Alternate source                |     |
| **SBNY**    | Signature Bank             | Still actually **Signature Bank** on Yahoo, now OTC. Yahoo recognizes the same company.                                                           | Alternate source  |     |
| **NFX**     | Newfield Exploration       | Old ticker disappeared after Encana acquisition; Yahoo still has old articles but no normal active NFX quote.                                     | Alternate source                |     |
| **STI**     | SunTrust Banks             | `STI` now means **Solidion Technology**, an unrelated company.                                                                                    | Alternate source                |     |
| **INFO**    | IHS Markit                 | `INFO` has been **reused**; there is now an ETF using INFO.                                                                                       | Alternate source                |     |
| **CSRA**    | CSRA Inc.                  | Old CSRA disappeared after acquisition; `CSRA` was **reused in Aug 2026** by a Cohen & Steers ETF.                                                | Alternate source                |     |
| **CAM**     | Cameron International      | Old ticker disappeared following the Schlumberger acquisition. Yahoo retains old CAM-related material but not the usable historical quote object. | Alternate source                |     |
| **BBBY**    | Original Bed Bath & Beyond | **Ticker reuse.** The current NYSE `BBBY` is the newer Bed Bath & Beyond/Beyond entity, not the bankrupt historical security in your dataset.     | Alternate source                |     |
| **TE**      | TECO Energy                | `TE` now belongs to **T1 Energy**, formerly FREYR Battery — completely unrelated to TECO.                                                         | Alternate source                |     |
| **EA**      | Electronic Arts            | Same company/ticker. Yahoo clearly still has long historical history ("All" return).                                                              | Alternate source  |     |
| **EQR**     | Equity Residential         | Same company/ticker and Yahoo still carries the security/history.                                                                                 | Alternate source  |     |
| **AVB**     | AvalonBay                  | Same company/ticker and Yahoo still has its long history.                                                                                         | Alternate source  |     |
| **SNDK**    | Old SanDisk                | `SNDK` has been **reused** by the newly independent Sandisk created in 2025. Yahoo describes the current company as incorporated in 2024.         | Alternate source                |     |
| **SCG**     | SCANA                      | Historical US `SCG` disappeared after the Dominion transaction; the old Yahoo identity is no longer normally retrievable by plain `SCG`.          | Alternate source                |     |


## 8. Replace unsuccessful Yahoo Finance downloads with Alternative source

### Historical ticker problems

`ticker_reused` contains symbols later assigned to a different company. `old_yahoo_ticker_disappeared` contains old symbols Yahoo no longer serves. Both groups are downloaded from Stooq only for the membership dates of the historical S&P 500 company.